# HMM Extension: Hidden Intent Modeling vs Markov Baseline

This notebook **extends** the existing Markov baseline by adding a **Hidden Markov Model (HMM)** layer.

Key idea:
- Markov baseline models observable transitions (Browse/Add_to_Cart/Purchase/Exit).
- HMM models **hidden intent states** that generate observable behavior patterns.


In [1]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from hmm.feature_engineering import (
    SessionConfig,
    add_state_column,
    build_event_level_features,
    reduced_feature_spec,
    select_reduced_features,
    sessionize_events,
)
from hmm.hmm_modeling import (
    HMMArtifacts,
    estimate_hidden_to_next_action,
    fit_scaler,
    hidden_state_profile_table,
    infer_hidden_states_viterbi,
    name_hidden_states,
    save_artifacts,
    train_gaussian_hmm,
)

ModuleNotFoundError: No module named 'hmm'

## STEP 1 — Feature engineering (20+ observable features)

We engineer temporal + behavioral features at the **event level** so the HMM can learn intent dynamics over time.

In [ ]:
csv_path = Path("../Data/events.csv")
raw = pd.read_csv(csv_path, usecols=["visitorid", "timestamp", "event"])
raw = raw[raw["event"].isin(["view", "addtocart", "transaction"])].copy()
raw["timestamp"] = pd.to_datetime(raw["timestamp"], unit="ms")
raw = raw.sort_values(["visitorid", "timestamp"]).reset_index(drop=True)

df = add_state_column(raw)
df = sessionize_events(df, SessionConfig(inactivity_threshold_minutes=30))

# Build full features, then reduce to a defensible 10–12 high-signal set
full_features_df, full_feature_cols = build_event_level_features(df, feature_set="full")
features_df, feature_cols, justification_df = select_reduced_features(full_features_df)

print("Full feature count:", len(full_feature_cols))
print("Reduced feature count:", len(feature_cols))
print("\nFinal reduced feature list:")
print(feature_cols)
print("\nJustification table:")
display(justification_df)

# Quick redundancy check on reduced set
corr = features_df.corr(numeric_only=True).abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
max_corr = upper.max().max()
print(f"\nMax absolute correlation within reduced set: {max_corr:.3f}")

features_df.head()

## STEP 2 — Define hidden states (3–5 intent states)

These states are **not directly observable**; they represent latent user intent.

In [ ]:
N_HIDDEN = 4
HIDDEN_STATE_IDEAS = [
    "Casual Browsing",
    "Interested",
    "High Purchase Intent",
    "Impulsive Buyer / Fast Conversion",
]
print("Hidden state count:", N_HIDDEN)
print("Intent labels (used after training):")
for s in HIDDEN_STATE_IDEAS:
    print("-", s)

## STEP 3 — Train HMM (Baum–Welch, unsupervised)

We train a Gaussian HMM on scaled feature vectors. Each session is a separate sequence.

In [ ]:
# STEP 3 (rigorous): session-level TRAIN/TEST split
session_sizes = df.groupby(["visitorid", "session_id"]).size().astype(int)
sessions = session_sizes.reset_index().rename(columns={0: "n_events"})
train_sessions, test_sessions = train_test_split(
    sessions,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)
train_keys = set(zip(train_sessions["visitorid"], train_sessions["session_id"]))

is_train = df.apply(lambda r: (r["visitorid"], r["session_id"]) in train_keys, axis=1)
df_train = df[is_train].copy().reset_index(drop=True)
df_test = df[~is_train].copy().reset_index(drop=True)

# Align features to the same rows (IMPORTANT: do not leak test into scaling)
full_train_features, _ = build_event_level_features(df_train, feature_set="full")
X_train_df, feature_cols, _ = select_reduced_features(full_train_features)

full_test_features, _ = build_event_level_features(df_test, feature_set="full")
X_test_df, _, _ = select_reduced_features(full_test_features)

train_lengths = df_train.groupby(["visitorid", "session_id"]).size().astype(int).tolist()
test_lengths = df_test.groupby(["visitorid", "session_id"]).size().astype(int).tolist()

scaler, X_train = fit_scaler(X_train_df.values.astype(float))
X_test = scaler.transform(X_test_df.values.astype(float))

# Train HMM on TRAIN sessions only (Baum–Welch)
hmm_model = train_gaussian_hmm(X_train, train_lengths, n_hidden_states=N_HIDDEN, n_iter=200)

print("Train log-likelihood:", hmm_model.score(X_train, train_lengths))
print("Test log-likelihood:", hmm_model.score(X_test, test_lengths))
print("Initial probabilities (pi):", np.round(hmm_model.startprob_, 4))
print("Hidden transition matrix (A):\n", np.round(hmm_model.transmat_, 4))

## STEP 4 — Inference (Viterbi + forward)

- **Viterbi** gives the most likely hidden intent sequence.
- The model score gives log-likelihood of sequences under the HMM.

In [ ]:
# Infer hidden intent sequences (Viterbi) on TRAIN and TEST separately
hidden_train = infer_hidden_states_viterbi(hmm_model, X_train, train_lengths)
hidden_test = infer_hidden_states_viterbi(hmm_model, X_test, test_lengths)

print("Hidden state IDs (train sample):", hidden_train[:20])
print("Hidden state IDs (test sample):", hidden_test[:20])

# Data-driven naming: first create a profile table on TRAIN, then assign names
train_profile = hidden_state_profile_table(
    df_events=df_train,
    features_df=X_train_df.assign(
        session_length_s=full_train_features["session_length_s"],
        engagement_score=full_train_features["engagement_score"],
        conversion_intent_score=full_train_features["conversion_intent_score"],
    ),
    hidden_states=hidden_train,
)

# Use mean conversion intent as an ordering signal, but interpret via the full profile table
hidden_names = name_hidden_states(
    hidden_train,
    conversion_intent_score=full_train_features["conversion_intent_score"],
    n_hidden=N_HIDDEN,
)

print("\nHidden-state validation table (TRAIN):")
train_profile_named = hidden_state_profile_table(
    df_events=df_train,
    features_df=full_train_features[["session_length_s", "engagement_score", "conversion_intent_score"]],
    hidden_states=hidden_train,
    hidden_state_names=hidden_names,
)
display(train_profile_named)

# Estimate mapping P(next observable | hidden) from TRAIN only
next_obs_train = df_train.groupby(["visitorid", "session_id"])["state"].shift(-1).fillna("Exit")
hidden_to_next = estimate_hidden_to_next_action(hidden_train, next_obs_train, n_hidden=N_HIDDEN)

display(hidden_to_next.round(4))

## STEP 5 — Comparison vs Markov baseline

We compare next-step prediction accuracy in a simple, fair way:
- **Markov** predicts next observable action from current observable state.
- **HMM** predicts next observable action from inferred hidden state via the learned mapping.

In [ ]:
def build_markov_matrix_from_states(state_series: pd.Series, next_state_series: pd.Series) -> pd.DataFrame:
    counts = pd.crosstab(state_series, next_state_series)
    if "Exit" in counts.index:
        counts = counts.drop(index="Exit")
    probs = counts.div(counts.sum(axis=1), axis=0)
    return probs.fillna(0.0)

# MARKOV baseline trained on TRAIN sessions only
next_train = df_train.groupby(["visitorid", "session_id"])["state"].shift(-1).fillna("Exit")
markov_probs = build_markov_matrix_from_states(df_train["state"], next_train)

# Evaluate on TEST sessions only
next_test = df_test.groupby(["visitorid", "session_id"])["state"].shift(-1).fillna("Exit")

# (A) Next-step accuracy: argmax next state
markov_preds = df_test["state"].map(lambda s: markov_probs.loc[s].idxmax() if s in markov_probs.index else "Exit")
markov_acc = (markov_preds.values == next_test.values).mean()

# HMM mapping is trained on TRAIN only: P(next observable | hidden)
hmm_next_map = hidden_to_next.copy()
hmm_top_next = hmm_next_map.idxmax(axis=1)

hidden_test_series = pd.Series(hidden_test, index=df_test.index)
hmm_preds = hidden_test_series.map(lambda hid: hmm_top_next.loc[hid] if hid in hmm_top_next.index else "Exit")
hmm_acc = (hmm_preds.values == next_test.values).mean()

# (B) Log-likelihood on TEST
# Markov log-likelihood: sum log P(next|current) on TEST transitions
markov_ll = 0.0
for curr, nxt in zip(df_test["state"].values, next_test.values):
    if curr in markov_probs.index and nxt in markov_probs.columns:
        p = float(markov_probs.loc[curr, nxt])
    else:
        p = 1e-12
    markov_ll += float(np.log(max(p, 1e-12)))

hmm_ll = float(hmm_model.score(X_test, test_lengths))

print(f"Markov next-step accuracy (TEST): {markov_acc:.4f}")
print(f"HMM next-step accuracy (TEST, mapped): {hmm_acc:.4f}")
print()
print(f"Markov log-likelihood (TEST transitions): {markov_ll:.2f}")
print(f"HMM log-likelihood (TEST sequences): {hmm_ll:.2f}")

## STEP 5B — Key differentiation example (same observable, different hidden intent)

We find two **real test sessions** where the user is currently in `Add_to_Cart`, but the HMM assigns **different** hidden intent states.

This is where HMM is strictly more expressive than a first-order Markov chain: Markov only conditions on the observable state, HMM conditions on **latent intent inferred from the full feature trajectory**.

In [ ]:
# Find 2 test sessions with same observable state but different hidden states

# Build a per-row key for test session
session_key_test = list(zip(df_test["visitorid"], df_test["session_id"]))

mask_cart = df_test["state"].eq("Add_to_Cart")
cart_rows = df_test[mask_cart].copy()
cart_hidden = pd.Series(hidden_test, index=df_test.index).loc[cart_rows.index]

# Group by session; pick first Add_to_Cart occurrence per session
first_cart_idx = cart_rows.groupby(["visitorid", "session_id"]).head(1).index
candidates = df_test.loc[first_cart_idx, ["visitorid", "session_id", "state"]].copy()
candidates["hidden_id"] = pd.Series(hidden_test, index=df_test.index).loc[first_cart_idx].values

# Look for two different hidden states
pair = None
for hid_a in sorted(candidates["hidden_id"].unique()):
    for hid_b in sorted(candidates["hidden_id"].unique()):
        if hid_a >= hid_b:
            continue
        a = candidates[candidates["hidden_id"] == hid_a].head(1)
        b = candidates[candidates["hidden_id"] == hid_b].head(1)
        if len(a) and len(b):
            pair = (a.iloc[0], b.iloc[0])
            break
    if pair:
        break

if pair is None:
    print("No differentiation pair found in test set (rare). Try N_HIDDEN=5 or adjust sessionization.")
else:
    a, b = pair
    markov_pred = markov_probs.loc["Add_to_Cart"].idxmax() if "Add_to_Cart" in markov_probs.index else "Exit"

    def hmm_pred_for_hidden(hid: int) -> str:
        return hidden_to_next.loc[hid].sort_values(ascending=False).idxmax() if hid in hidden_to_next.index else "Exit"

    rows = []
    for row in [a, b]:
        hid = int(row["hidden_id"])
        rows.append(
            {
                "User": int(row["visitorid"]),
                "Session": int(row["session_id"]),
                "Current State": row["state"],
                "Markov Prediction": markov_pred,
                "HMM Hidden State": hidden_names[hid] if hid < len(hidden_names) else f"hidden_{hid}",
                "HMM Prediction": hmm_pred_for_hidden(hid),
            }
        )

    diff_table = pd.DataFrame(rows)
    display(diff_table)

    print("\nExplanation:")
    print("- Markov uses only current observable state → both rows get the SAME prediction.")
    print("- HMM infers a hidden intent state from the trajectory of engineered features → predictions differ.")

## STEP 6 — Visualization improvements

We generate:
- Markov transition heatmap (train)
- HMM hidden transition heatmap
- Hidden state distribution (test)
- Example session plot: observed vs hidden states

In [ ]:
# Markov heatmap (TRAIN)
plt.figure(figsize=(7, 5))
sns.heatmap(markov_probs, annot=True, fmt=".3f", cmap="Blues")
plt.title("Markov Transition Heatmap (TRAIN)")
plt.tight_layout()
plt.show()

# HMM hidden heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(hmm_model.transmat_, annot=True, fmt=".3f", cmap="Blues")
plt.title("HMM Hidden-State Transition Heatmap")
plt.tight_layout()
plt.show()

# Hidden distribution (TEST)
hidden_dist_test = pd.Series(hidden_test).value_counts(normalize=True).sort_index()
ax = hidden_dist_test.plot(kind="bar", title="Hidden State Distribution (TEST)")
ax.set_ylabel("Proportion")
plt.tight_layout()
plt.show()

In [ ]:
# Example sequence plot: observed vs hidden (pick one test session)

# Pick a test session with at least 15 events
sess_counts = df_test.groupby(["visitorid", "session_id"]).size().sort_values(ascending=False)
vid, sid = sess_counts.index[0]
sess = df_test[(df_test["visitorid"] == vid) & (df_test["session_id"] == sid)].copy()

# Align hidden states for this session
hidden_test_series = pd.Series(hidden_test, index=df_test.index)
sess_hidden = hidden_test_series.loc[sess.index].values

obs_map = {"Browse": 0, "Add_to_Cart": 1, "Purchase": 2, "Exit": 3}
obs_numeric = sess["state"].map(obs_map).values

plt.figure(figsize=(10, 3))
plt.plot(obs_numeric, marker="o", label="Observed state (coded)")
plt.plot(sess_hidden, marker="s", label="Hidden state id")
plt.yticks([0, 1, 2, 3], ["Browse", "Add_to_Cart", "Purchase", "Exit"])
plt.title(f"Observed vs Hidden (test session: visitor={int(vid)}, session={int(sid)})")
plt.xlabel("Event index")
plt.legend()
plt.tight_layout()
plt.show()

## STEP 8 — Final comparison summary

| Aspect | Markov Chain | Hidden Markov Model |
|---|---|---|
| Observability | Uses only observable states (Browse/Add_to_Cart/Purchase/Exit) | Introduces latent intent states inferred from observables |
| Memory | First-order: depends only on current observable state | Has implicit memory via latent state sequence and emissions |
| Interpretability | High at the action level | High when hidden states are **validated with behavior metrics** |
| Accuracy | Strong baseline for next-action prediction | Can improve by separating intent groups inside the same observable state |
| Real-world usefulness | Good for funnel transitions + simple prediction | Better for personalization, risk scoring, and intent-aware interventions |

## STEP 6 — Visualizations

- Markov heatmap: already in your baseline notebook.
- HMM hidden-state transition heatmap.
- Hidden state distribution.
- Sample journey: observed vs hidden.

In [ ]:
# HMM hidden transition heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(hmm_model.transmat_, annot=True, fmt=".3f", cmap="Blues")
plt.title("HMM Hidden-State Transition Heatmap")
plt.tight_layout()
plt.show()

# Hidden state distribution
hidden_dist = pd.Series(hidden_viterbi).value_counts(normalize=True).sort_index()
hidden_dist.plot(kind="bar", title="Hidden State Distribution")
plt.ylabel("Proportion")
plt.tight_layout()
plt.show()

In [ ]:
# Sample journey: observed states vs hidden intent
sample_vid = df["visitorid"].iloc[0]
sample = df[df["visitorid"] == sample_vid].head(30).copy()
sample_hidden = hidden_ids.loc[sample.index].values

print("Sample visitor:", sample_vid)
print(pd.DataFrame({
    "timestamp": sample["timestamp"].values,
    "observed_state": sample["state"].values,
    "hidden_state_id": sample_hidden,
    "hidden_state_name": [hidden_names[i] for i in sample_hidden],
}).head(15))

## STEP 8 — Save artifacts for Streamlit

We save the HMM model, scaler, state names, and the hidden→next-action mapping.
The Streamlit app can load these **without recomputing** the model.

In [ ]:
app_dir = Path("../app")
artifacts = HMMArtifacts(
    model=hmm_model,
    scaler=scaler,
    hidden_state_names=hidden_names,
    hidden_to_next_action=hidden_to_next,
)
save_artifacts(app_dir, artifacts)
print("Saved HMM artifacts to:", app_dir.resolve())